# Урок 08 - Шаблон за дизайн с множество агенти


## Настройка


In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

%pip install agent-framework azure-ai-projects azure-identity --quiet

import os
import asyncio

from agent_framework import AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Защо мултиагентни системи?

Задачи от реалния свят, като планиране на пътуване, изискват различни видове експертиза — логистика, местни знания, бюджетиране и други. Един агент, опитващ се да се справи с всичко, бързо става непоследователен.

Мултиагентните системи решават това чрез **специализация**: всеки агент се фокусира върху една област на експертиза, произвеждайки резултати с по-високо качество в сравнение с генералист. Те също така подобряват **скалируемостта** — можете да добавяте нови агенти (например специалист по полети, ресторантен критик), без да преписвате съществуващия работен процес. Агентите се свързват чрез структурирана последователност, предавайки контекста от един към друг.


## Създаване на специализирани агенти


In [ ]:
planner_agent = await provider.create_agent(
    name="TravelPlanner",
    instructions="You are a travel planning specialist. Create detailed trip itineraries based on the traveler's preferences. Include daily schedules, must-see attractions, and logistical tips.",
)

concierge_agent = await provider.create_agent(
    name="TravelConcierge",
    instructions="You are a travel concierge who reviews and enhances trip plans. Review the plan for completeness, add local insider tips, suggest restaurants, and identify potential issues. Provide your feedback in a constructive format.",
)

## Създаване на последователен работен процес

`WorkflowBuilder` ви позволява да свържете агенти в ориентиран граф. Тук създаваме прост двустепенен процес: **TravelPlanner** изготвя маршрута, след което **TravelConcierge** го преглежда и подобрява.


In [ ]:
workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .build()

last_author = None
events = workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Добавяне на още агенти към работния процес

Едно от най-големите предимства на модела с няколко агента е колко лесно се разширява. По-долу добавяме агент **BudgetReviewer**, който проверява плана спрямо бюджета на пътуващия, отбелязва елементи, които биха могли да надхвърлят разходите, и предлага алтернативи за спестяване на пари. Работният процес сега изпълнява три агенти последователно:

```
TravelPlanner → TravelConcierge → BudgetReviewer
```


In [ ]:
budget_agent = await provider.create_agent(
    name="BudgetReviewer",
    instructions="You are a budget-conscious travel advisor. Review the proposed trip plan and concierge enhancements against the traveler's stated budget. Estimate costs for flights, hotels, meals, and activities. Flag anything that risks exceeding the budget and suggest cost-saving alternatives while preserving the trip's quality.",
)

extended_workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .add_edge(concierge_agent, budget_agent) \
    .build()

last_author = None
events = extended_workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Обобщение

В този урок научихте как да:

1. **Създавате специализирани агенти** — всеки със своя фокусирана роля (планиране, консиерж, преглед на бюджета).
2. **Свържете агентите в последователен работен процес** чрез `WorkflowBuilder` и `add_edge`.
3. **Излъчвате изход от многократен агентски конвейер**, проследявайки кой агент говори.
4. **Разширявате работния процес** като добавяте нови агенти към веригата без да променяте съществуващите.

Дизайн шаблонът с множество агенти поддържа всеки агент прост, докато произвежда по-богати и по-задълбочено прегледани резултати, отколкото който и да е единичен агент може да постигне самостоятелно.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Отказ от отговорност**:
Този документ е преведен с помощта на AI преводаческа услуга [Co-op Translator](https://github.com/Azure/co-op-translator). Въпреки че се стремим към точност, моля, имайте предвид, че автоматизираните преводи могат да съдържат грешки или неточности. Оригиналният документ на неговия език трябва да се счита за авторитетен източник. За критична информация се препоръчва професионален човешки превод. Ние не носим отговорност за каквито и да било недоразумения или погрешни тълкувания, произтичащи от използването на този превод.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
